# 06 — Feature engineering

Runs `src/features/engineering.engineer_features`, ported from
`feature-engineering.ipynb`. Input `gurgaon_properties_cleaned_v1.csv`
(3 803, 17); output matches `gurgaon_properties_cleaned_v2.csv` (3 803, 23) —
the raw text columns (`areaWithType`, `additionalRoom`, `furnishDetails`,
`features`, `nearbyLocations`) are parsed into numeric features and dropped.

**Match: 22 / 23 columns exact.** The one that differs is `sector` — the module
passes it through from `cleaned_v1` untouched, but the committed `cleaned_v2`
carries a *normalised* version. 158 rows differ, across **two distinct
patterns** (broken out below): ~138 cosmetic (suffix strip / doubled word) and
20 substantive (`sector 3 phase 2` / `sector 3 phase 3 extension` → `sector 5`).
That normalisation step is not in `feature-engineering.ipynb` — a small untraced
gap, same family as the ones in notebooks 02 and 04.

A demonstration of already-tested code — no new logic.

In [1]:
import sys, logging
from pathlib import Path

REPO_ROOT = Path.cwd().parents[0] if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

from src.features.engineering import engineer_features

INTERIM = REPO_ROOT / "data" / "interim"
RAW = REPO_ROOT / "data" / "raw"


## What `engineer_features` does

| step | produces | from |
|---|---|---|
| `engineer_area_columns` | `super_built_up_area`, `built_up_area`, `carpet_area` | the `areaWithType` free text, with a unit-scale fix for plot-style rows |
| `engineer_additional_room_flags` | `study room`, `servant room`, `store room`, `pooja room`, `others` (0/1) | the `additionalRoom` text |
| `categorize_age_possession` | `agePossession` re-bucketed (`"New Property"`, `"Relatively New"`, …, `"Undefined"`) | the free-text `agePossession` |
| `engineer_furnishing_type` | `furnishing_type` (0 / 1 / 2) | KMeans (`random_state=42`) on per-item furnishing counts parsed from `furnishDetails` |
| `engineer_luxury_score` | `luxury_score` | the `features` amenity list weighted by `LUXURY_WEIGHTS`; missing `features` backfilled from `appartments.csv` via a society-name join |

Then `nearbyLocations`, `furnishDetails`, `features`, `additionalRoom` are dropped.

In [2]:
v1 = pd.read_csv(INTERIM / "gurgaon_properties_cleaned_v1.csv")
appartments = pd.read_csv(RAW / "appartments.csv")
print("input      :", v1.shape)
print("appartments:", appartments.shape, "(for the luxury_score backfill join)")


input      : (3803, 17)
appartments: (247, 7) (for the luxury_score backfill join)


In [3]:
engineered = engineer_features(v1.copy(), appartments_df=appartments.copy())
print("\ninput  :", v1.shape)
print("output :", engineered.shape)


engineer_area_columns: recovered built_up_area for 546 plot-style rows


engineer_luxury_score: backfilled features for 154/635 rows via appartments join



input  : (3803, 17)
output : (3803, 23)


In [4]:
engineered.head()


,property_type,society,sector,price,price_per_sqft,area,areaWithType,bedRoom,bathroom,balcony,...,super_built_up_area,built_up_area,carpet_area,study room,servant room,store room,pooja room,others,furnishing_type,luxury_score
0,flat,signature global park 4,sector 36,0.82,7585.0,1081.0,Super Built up area 1081(100.43 sq.m.)Carpet a...,3,2,2,...,1081.0,NaN,650.0,0,0,0,0,0,0,8
1,flat,smart world gems,sector 89,0.95,8600.0,1105.0,Carpet area: 1103 (102.47 sq.m.),2,2,2,...,NaN,NaN,1103.0,1,1,0,0,0,0,38
2,flat,pyramid elite,sector 86,0.46,79.0,58228.0,Carpet area: 58141 (5401.48 sq.m.),2,2,1,...,NaN,NaN,58141.0,0,0,0,0,0,0,15
3,flat,breez global hill view,sohna road,0.32,5470.0,585.0,Built Up area: 1000 (92.9 sq.m.)Carpet area: 5...,2,2,1,...,NaN,1000.0,585.0,0,0,0,0,0,0,49
4,flat,bestech park view sanskruti,sector 92,1.60,8020.0,1995.0,Super Built up area 1995(185.34 sq.m.)Built Up...,3,4,3+,...,1995.0,1615.0,1476.0,0,1,0,0,1,1,174


In [5]:
expected = pd.read_csv(INTERIM / "gurgaon_properties_cleaned_v2.csv")
assert list(engineered.columns) == list(expected.columns)
assert engineered.shape == expected.shape
exact, diffs = [], {}
for col in expected.columns:
    a, b = engineered[col], expected[col]
    m = (np.isclose(pd.to_numeric(a, errors="coerce").astype(float), b.astype(float), equal_nan=True)
         if b.dtype.kind in "fi" else a.astype(str) == b.astype(str))
    if m.all():
        exact.append(col)
    else:
        diffs[col] = int((~m).sum())
print(f"exact columns: {len(exact)}/{len(expected.columns)}")
print(f"columns that differ: {diffs}")


exact columns: 22/23
columns that differ: {'sector': 158}


## The one deviation — `sector`

`engineer_features` never touches `sector`; it is passed straight through from
`cleaned_v1` (asserted below). The committed `cleaned_v2` normalises it via an
untraced step. Categorising all 158 differing rows splits them cleanly:

- **cosmetic (~138 rows)** — a trailing letter stripped off the same base
  number (`sector 37c → sector 37`, `99a → 99`, `17a`/`17b → 17`), or a doubled
  word collapsed (`sohna road road → sohna road`).
- **substantive (20 rows, 2 source strings)** — `sector 3 phase 2` **and**
  `sector 3 phase 3 extension` are both remapped to `sector 5`. A genuine
  reclassification of that pocket, not a suffix trim.

Only 11 distinct `(from → to)` pairs in total.

In [6]:
import re

sec_mm = engineered["sector"].astype(str) != expected["sector"].astype(str)
print(int(sec_mm.sum()), "rows differ; engineered == cleaned_v1 for all of them:",
      bool((engineered.loc[sec_mm, "sector"].astype(str).values
            == v1.loc[sec_mm, "sector"].astype(str).values).all()))

pairs = pd.DataFrame({"from": v1.loc[sec_mm, "sector"].astype(str).values,
                      "to": expected.loc[sec_mm, "sector"].astype(str).values})

def categorise(a, b):
    if re.match(r"^(sector \d+)[a-z]$", a) and re.match(r"^(sector \d+)[a-z]$", a).group(1) == b:
        return "cosmetic: suffix-letter"
    if re.sub(r"\b(\w[\w ]*?)\s+\1\b", r"\1", a) == b:
        return "cosmetic: doubled-word"
    return "substantive remap"

pairs["category"] = [categorise(a, b) for a, b in zip(pairs["from"], pairs["to"])]
print("\nby row count:")
print(pairs["category"].value_counts().to_string())
pairs.drop_duplicates().sort_values("category")


158 rows differ; engineered == cleaned_v1 for all of them: True

by row count:
category
cosmetic: suffix-letter    126
substantive remap           20
cosmetic: doubled-word      12


,from,to,category
15,sohna road road,sohna road,cosmetic: doubled-word
0,sector 37c,sector 37,cosmetic: suffix-letter
1,sector 36a,sector 36,cosmetic: suffix-letter
2,sector 17a,sector 17,cosmetic: suffix-letter
4,sector 99a,sector 99,cosmetic: suffix-letter
5,sector 9a,sector 9,cosmetic: suffix-letter
23,sector 10a,sector 10,cosmetic: suffix-letter
24,sector 17b,sector 17,cosmetic: suffix-letter
82,sector 88b,sector 88,cosmetic: suffix-letter
10,sector 3 phase 3 extension,sector 5,substantive remap


## The engineered columns

### Areas — `areaWithType` → three numbers

`engineer_area_columns` parses `super_built_up_area` / `built_up_area` /
`carpet_area` out of the `areaWithType` string, and `_fix_unit_scale` rescales
plot-style rows recorded in sq. yards / sq. metres (the log line above:
"recovered built_up_area for ~546 plot-style rows").

In [7]:
cols = ["areaWithType", "super_built_up_area", "built_up_area", "carpet_area"]
engineered[cols].head(8)


,areaWithType,super_built_up_area,built_up_area,carpet_area
0,Super Built up area 1081(100.43 sq.m.)Carpet a...,1081.0,NaN,650.0
1,Carpet area: 1103 (102.47 sq.m.),NaN,NaN,1103.0
2,Carpet area: 58141 (5401.48 sq.m.),NaN,NaN,58141.0
3,Built Up area: 1000 (92.9 sq.m.)Carpet area: 5...,NaN,1000.0,585.0
4,Super Built up area 1995(185.34 sq.m.)Built Up...,1995.0,1615.0,1476.0
5,Super Built up area 632(58.71 sq.m.)Carpet are...,632.0,NaN,532.0
6,Super Built up area 5350(497.03 sq.m.),5350.0,NaN,NaN
7,Super Built up area 2338(217.21 sq.m.),2338.0,NaN,NaN


### Room flags, age bucket, furnishing, luxury

In [8]:
print("additionalRoom -> five 0/1 flags:")
print(pd.concat([v1["additionalRoom"].head(6),
                 engineered[["study room", "servant room", "store room",
                             "pooja room", "others"]].head(6)], axis=1).to_string())

print("\nagePossession  (raw -> bucketed):")
print("  raw distinct   :", v1["agePossession"].nunique())
print("  after bucketing:", engineered["agePossession"].value_counts().to_dict())

print("\nfurnishing_type (KMeans, random_state=42):",
      engineered["furnishing_type"].value_counts().sort_index().to_dict())

print("\nluxury_score    :", engineered["luxury_score"].describe()[["min", "50%", "max"]].to_dict())


additionalRoom -> five 0/1 flags:
            additionalRoom  study room  servant room  store room  pooja room  others
0            not available           0             0           0           0       0
1  study room,servant room           1             1           0           0       0
2            not available           0             0           0           0       0
3            not available           0             0           0           0       0
4      servant room,others           0             1           0           0       1
5               store room           0             0           1           0       0

agePossession  (raw -> bucketed):
  raw distinct   : 48
  after bucketing: {'Relatively New': 1676, 'New Property': 626, 'Moderately Old': 575, 'Undefined': 333, 'Old Property': 310, 'Under Construction': 283}

furnishing_type (KMeans, random_state=42): {0: 2509, 1: 1078, 2: 216}

luxury_score    : {'min': 0.0, '50%': 58.0, 'max': 174.0}


## Two deliberate changes from the notebook (documented in the module)

1. **`_drop_temp_furnishing_columns` drops by column name**, not the notebook's
   `df.iloc[:, :-18]`. The positional slice assumes exactly 18 distinct
   furnishing items are found; one different amenity name on a fresh scrape and
   it silently drops the wrong columns.
2. **`_label_furnishing_clusters` orders the KMeans labels by mean furnishing
   count** rather than trusting them to come out `0 = unfurnished / 1 = semi /
   2 = furnished`. KMeans doesn't guarantee cluster-index order; the notebook's
   mapping was right for that one run but not guaranteed to hold if the data,
   sklearn version, or furnishing vocabulary changes.

Neither changes the output on this data — the 22/23 match holds — they just
make the step survive a different input.